# Fundamentos de Inteligencia Artificial
## Unidad II — Espacios de búsqueda

Este notebook es **solo una demostración para clase**.  
El objetivo es observar cómo un problema de búsqueda puede representarse en Python mediante:

- **Estados** → nodos.
- **Acciones o transiciones** → conexiones.
- **Estado inicial** → punto de partida.
- **Meta** → estado objetivo.

> En esta muestra todavía no estudiaremos formalmente BFS, DFS ni heurísticas. Primero nos interesa aprender a **representar el espacio de búsqueda**.


## Preparación

Usaremos `networkx` para construir grafos y `matplotlib` para visualizarlos.  
Google Colab normalmente ya incluye ambas bibliotecas.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

print("Bibliotecas cargadas correctamente.")

# Ejemplo 1. Robot dentro de un edificio

Un robot puede desplazarse entre:

- Entrada
- Pasillo
- Aula
- Laboratorio

Las conexiones indican **movimientos permitidos**.

**Inicio:** Entrada  
**Meta:** Laboratorio


In [ ]:
# Representamos las conexiones del edificio
grafo = {
    "Entrada": ["Pasillo"],
    "Pasillo": ["Entrada", "Aula", "Laboratorio"],
    "Aula": ["Pasillo"],
    "Laboratorio": ["Pasillo"]
}

# Creamos el grafo
G = nx.Graph()

for estado, vecinos in grafo.items():
    for vecino in vecinos:
        G.add_edge(estado, vecino)

print("Estados:", list(G.nodes()))
print("Transiciones:", list(G.edges()))

In [ ]:
# Posiciones fijas para que el dibujo sea fácil de interpretar
pos = {
    "Entrada": (0, 0),
    "Pasillo": (1, 0),
    "Aula": (2, 1),
    "Laboratorio": (2, -1)
}

plt.figure(figsize=(8, 4))
nx.draw(G, pos, with_labels=True, node_size=2500, font_size=10)
plt.title("Espacio de búsqueda: robot en un edificio")
plt.show()

### ¿Qué representa Python?

- Cada **nodo** es un estado.
- Cada **línea** es una transición permitida.
- La ruta `Entrada → Pasillo → Laboratorio` es un camino dentro del espacio.
- El espacio contiene también estados que quizá no formen parte de la solución final.


# Ejemplo 2. Ciudades conectadas

Ahora representaremos un pequeño mapa.

**Inicio:** Querétaro  
**Meta:** Salamanca

La idea es la misma: las ciudades son estados y las carreteras son transiciones.


In [ ]:
ciudades = {
    "Querétaro": ["Celaya", "San Juan del Río"],
    "Celaya": ["Querétaro", "Salamanca"],
    "San Juan del Río": ["Querétaro", "Tequisquiapan"],
    "Tequisquiapan": ["San Juan del Río", "Salamanca"],
    "Salamanca": ["Celaya", "Tequisquiapan"]
}

G2 = nx.Graph()

for ciudad, conectadas in ciudades.items():
    for destino in conectadas:
        G2.add_edge(ciudad, destino)

print("Número de estados:", G2.number_of_nodes())
print("Número de conexiones:", G2.number_of_edges())

In [ ]:
pos2 = nx.spring_layout(G2, seed=7)

plt.figure(figsize=(9, 5))
nx.draw(G2, pos2, with_labels=True, node_size=2600, font_size=9)
plt.title("Espacio de búsqueda: ciudades conectadas")
plt.show()

### Observación

Hay más de una forma de llegar a Salamanca:

- Querétaro → Celaya → Salamanca
- Querétaro → San Juan del Río → Tequisquiapan → Salamanca

El **espacio de búsqueda** contiene ambas posibilidades.  
Más adelante veremos cómo decide un algoritmo qué camino explorar primero.


# Ejemplo 3. Estados numéricos

No todos los estados representan lugares.

Supongamos:

- **Inicio:** 0
- **Acciones:** sumar 1 o sumar 2
- **Meta:** 4

Cada número es un estado.


In [ ]:
inicio = 0
meta = 4

G3 = nx.DiGraph()

for estado in range(0, meta + 1):
    if estado + 1 <= meta:
        G3.add_edge(estado, estado + 1, accion="+1")
    if estado + 2 <= meta:
        G3.add_edge(estado, estado + 2, accion="+2")

print("Estados:", list(G3.nodes()))
print("Transiciones:")
for origen, destino, datos in G3.edges(data=True):
    print(f"{origen} -> {destino}  ({datos['accion']})")

In [ ]:
pos3 = {
    0: (0, 0),
    1: (1, 1),
    2: (1, -1),
    3: (2, 0.7),
    4: (3, 0)
}

plt.figure(figsize=(8, 5))
nx.draw(G3, pos3, with_labels=True, node_size=2200, arrows=True)
labels = nx.get_edge_attributes(G3, "accion")
nx.draw_networkx_edge_labels(G3, pos3, edge_labels=labels, font_size=9)
plt.title("Espacio de búsqueda numérico")
plt.show()

### Idea importante

El estado `2` puede alcanzarse de diferentes maneras:

- `0 → 2`
- `0 → 1 → 2`

Por eso, un mismo estado puede pertenecer a diferentes caminos.


# Ejemplo 4. Dos interruptores

Tenemos dos interruptores: A y B.

Cada uno puede estar:

- `0` = apagado
- `1` = encendido

Los estados posibles son:

- `00`
- `01`
- `10`
- `11`

Una acción consiste en **cambiar solamente un interruptor**.


In [ ]:
estados = ["00", "01", "10", "11"]

G4 = nx.Graph()

for estado in estados:
    G4.add_node(estado)

# Conectar estados que difieren en un solo bit
for i, e1 in enumerate(estados):
    for e2 in estados[i+1:]:
        diferencias = sum(a != b for a, b in zip(e1, e2))
        if diferencias == 1:
            G4.add_edge(e1, e2)

print("Estados:", list(G4.nodes()))
print("Transiciones:", list(G4.edges()))

In [ ]:
pos4 = {
    "00": (0, 0),
    "01": (1, 0),
    "10": (0, 1),
    "11": (1, 1)
}

plt.figure(figsize=(6, 6))
nx.draw(G4, pos4, with_labels=True, node_size=2500, font_size=12)
plt.title("Espacio de búsqueda: dos interruptores")
plt.show()

### Lo que cambia aquí

Un nodo ya no representa una ciudad ni una habitación.  
Representa una **configuración**.

Por ejemplo:

`10` = A encendido y B apagado.


# Ejemplo 5. Jarras de 4 L y 3 L

Ahora usamos un ejemplo más completo.

Un estado se representa como:

`(x, y)`

donde:

- `x` = litros en la jarra de 4 L
- `y` = litros en la jarra de 3 L

**Inicio:** `(0, 0)`  
**Meta:** cualquier estado donde alguna jarra tenga exactamente 2 litros.

Las acciones válidas son:

- llenar una jarra;
- vaciar una jarra;
- pasar agua de una jarra a la otra.

En este ejemplo Python generará automáticamente los estados alcanzables.


In [ ]:
from collections import deque

CAP4 = 4
CAP3 = 3
inicio = (0, 0)

def sucesores(estado):
    x, y = estado
    posibles = set()

    # Llenar
    posibles.add((CAP4, y))
    posibles.add((x, CAP3))

    # Vaciar
    posibles.add((0, y))
    posibles.add((x, 0))

    # Verter de 4L hacia 3L
    cantidad = min(x, CAP3 - y)
    posibles.add((x - cantidad, y + cantidad))

    # Verter de 3L hacia 4L
    cantidad = min(y, CAP4 - x)
    posibles.add((x + cantidad, y - cantidad))

    posibles.discard(estado)
    return posibles

# Generar todos los estados alcanzables desde (0,0)
visitados = {inicio}
pendientes = deque([inicio])
G5 = nx.DiGraph()

while pendientes:
    actual = pendientes.popleft()

    for siguiente in sucesores(actual):
        G5.add_edge(actual, siguiente)

        if siguiente not in visitados:
            visitados.add(siguiente)
            pendientes.append(siguiente)

metas = [e for e in visitados if 2 in e]

print("Estados alcanzables:", sorted(visitados))
print("Número de estados alcanzables:", len(visitados))
print("Estados que cumplen la meta:", sorted(metas))

In [ ]:
plt.figure(figsize=(12, 8))
pos5 = nx.spring_layout(G5, seed=4)

nx.draw(
    G5,
    pos5,
    with_labels=True,
    node_size=1400,
    font_size=8,
    arrows=True,
    arrowsize=12
)

plt.title("Espacio de búsqueda alcanzable: jarras de 4 L y 3 L")
plt.show()

### ¿Qué podemos observar?

Python no recibió una lista completa de estados.

En su lugar:

1. comenzó en `(0,0)`;
2. aplicó las acciones permitidas;
3. generó nuevos estados;
4. volvió a aplicar acciones;
5. construyó el espacio alcanzable.

Esto muestra por qué las **acciones y restricciones** son tan importantes: determinan qué estados pueden formar parte del espacio de búsqueda.


# Resumen

En todos los ejemplos ocurrió lo mismo:

| Concepto | Representación en Python |
|---|---|
| Estado | Nodo |
| Acción / transición | Conexión |
| Estado inicial | Nodo de partida |
| Meta | Nodo o condición objetivo |
| Espacio de búsqueda | Conjunto de nodos y conexiones |
| Solución | Camino válido desde inicio hasta meta |

## Pregunta para cerrar la demostración

Ya podemos representar el espacio de búsqueda.

**¿Cómo decide ahora la computadora qué nodo explorar primero?**

Esa pregunta nos llevará a los **algoritmos de búsqueda**.
